# 03. v0 정식 평가

`eval/testset.csv` 110문항으로 v0(텍스트 RAG)의 기준 점수를 잽니다.
채점 규칙은 `docs/evaluation_guide.md`, 채점 코드는 `eval/rag_eval.py`에 있습니다.

- 인덱스는 2주차에 만든 `data/index/v0_faiss`를 **불러오기만** 합니다. (임베딩 비용 0)
- **API 사용 (요금 발생)** 칸: ④ 평가 실행 (예상 약 $0.1)

## ① 설정
`.env`를 읽고, 프로젝트 폴더를 import 경로에 추가해 `eval/rag_eval.py`를 불러옵니다.

In [1]:
import os
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore", message=".*langchain-community.*")

from dotenv import load_dotenv
import pandas as pd
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.callbacks import get_usage_metadata_callback

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(PROJECT_DIR / ".env")
sys.path.insert(0, str(PROJECT_DIR))  # eval/rag_eval.py를 불러오기 위해

from eval.rag_eval import load_testset, run_eval, summarize, JudgeResult

LLM_MODEL = os.getenv("LLM_MODEL")
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL")
JUDGE_MODEL = os.getenv("JUDGE_MODEL")
TOP_K = 5
RESULTS_DIR = PROJECT_DIR / "eval" / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("API 키:", "있음" if os.getenv("OPENAI_API_KEY") else "없음 (.env 확인)")
print("답변 모델:", LLM_MODEL, "| 채점 모델:", JUDGE_MODEL, "| 임베딩:", EMBEDDING_MODEL, "| TOP_K:", TOP_K)

API 키: 있음
답변 모델: gpt-6-luna | 채점 모델: gpt-6.1-terra | 임베딩: text-embedding-3-small | TOP_K: 5


## ② v0 불러오기
2주차 인덱스와 **같은 프롬프트**로 답하는 함수를 만듭니다. 평가를 위해 답변과 함께 **검색된 (문서, 쪽) 목록**도 돌려줍니다.

In [2]:
INDEX_DIR = PROJECT_DIR / "data" / "index" / "v0_faiss"
vectorstore = FAISS.load_local(str(INDEX_DIR), OpenAIEmbeddings(model=EMBEDDING_MODEL),
                               allow_dangerous_deserialization=True)  # 직접 만든 파일이므로 안전
llm = ChatOpenAI(model=LLM_MODEL)

# 02_text_rag_v0와 같은 프롬프트 (버전 간 비교를 위해 바꾸지 않음)
PROMPT = """당신은 통계 보고서 내용으로만 답하는 도우미입니다.
규칙:
1. 아래 [근거]에 있는 내용으로만 답하세요. 추측하지 마세요.
2. 답의 끝에 사용한 근거를 [문서ID p.쪽] 형식으로 표시하세요. 예: [D01 p.8]
3. [근거]에서 답을 찾을 수 없으면 정확히 "모름"이라고만 답하세요.

[근거]
{context}

[질문]
{question}"""

def answer_v0(question):
    found = vectorstore.similarity_search(question, k=TOP_K)
    context = "\n\n".join(f"[{d.metadata['doc_id']} p.{d.metadata['page']}]\n{d.page_content}" for d in found)
    response = llm.invoke(PROMPT.format(context=context, question=question))
    return response.text, [(d.metadata["doc_id"], d.metadata["page"]) for d in found]

print("인덱스 청크 수:", vectorstore.index.ntotal)

인덱스 청크 수: 264


## ③ 테스트셋

In [3]:
testset = load_testset(PROJECT_DIR / "eval" / "testset.csv")
print("문항 수:", len(testset))
testset.groupby(["type", "chart_kind"]).size()

문항 수: 110


type     chart_kind
chart    image         12
         vector        18
table                  30
text                   40
unknown                10
dtype: int64

## ④ 평가 실행 — API 사용 (요금 발생)
110문항에 답하고 채점해 `eval/results/v0.csv`로 저장합니다. 사용한 토큰도 모델별로 기록합니다.

In [4]:
RESULT_PATH = RESULTS_DIR / "v0.csv"
judge_llm = ChatOpenAI(model=JUDGE_MODEL).with_structured_output(JudgeResult)

if RESULT_PATH.exists():
    # 이미 평가한 결과가 있으면 다시 API를 부르지 않음 (요금 0, 결과도 그대로 유지)
    # 다시 평가하려면 eval/results/v0.csv를 지우고 실행
    results = pd.read_csv(RESULT_PATH, dtype={"chart_kind": str}).fillna({"chart_kind": "", "judge_reason": ""})
    results["recall_hit"] = results["recall_hit"].map({True: True, False: False, "True": True, "False": False})
    print("저장된 결과를 불러왔습니다 (API 호출 없음):", RESULT_PATH)
else:
    with get_usage_metadata_callback() as usage:
        results = run_eval(testset, answer_v0, judge_llm, k=TOP_K)
    results.to_csv(RESULT_PATH, index=False)
    print("저장:", RESULT_PATH)
    for model, u in usage.usage_metadata.items():
        print(f"{model}: 입력 {u['input_tokens']:,} / 출력 {u['output_tokens']:,} 토큰")

저장된 결과를 불러왔습니다 (API 호출 없음): /home/user/git_project/project6/ReportQA/eval/results/v0.csv


## ⑤ 요약
README 9장 표에 들어갈 값입니다. (0.82 = 82%)
- Recall@5: 정답 쪽이 검색 결과 5개 안에 있었던 비율 (모름 문항 제외 100문항)
- 환각률: 모름 문항 10개 중 답을 지어낸 비율

In [5]:
summary = summarize(results)
print("판정 분포:", results["judge"].value_counts().to_dict())
summary

판정 분포: {'correct': 74, 'refused': 28, 'wrong': 8}


Recall@5      0.870
정답률           0.673
텍스트           0.925
표             0.533
차트(vector)    0.611
차트(image)     0.000
환각률           0.000
dtype: float64

## ⑥ 실패 사례
유형별로 틀린 문항을 봅니다. 검색이 정답 쪽을 찾았는지(`recall_hit`)를 함께 보면 실패 원인을 나눌 수 있습니다.
- `recall_hit=False` → 검색 실패 (6주차 검색 개선 대상)
- `recall_hit=True`인데 틀림 → 찾았지만 읽지 못함 (표 구조·차트 문제 → v1·v2 대상)

In [6]:
pd.set_option("display.max_colwidth", 80)
view = results.merge(testset[["id", "question", "answer", "answer_pages", "doc_id"]], on="id")
failed = view[view["judge"] != "correct"]
print(pd.crosstab(failed["type"], failed["recall_hit"].map({True: "검색 성공", False: "검색 실패", None: "-"})))
failed.groupby("type").head(3)[["id", "question", "answer", "answer_text", "recall_hit", "judge"]]

recall_hit  검색 성공  검색 실패
type                    
chart          13      6
table           8      6
text            3      0


,id,question,answer,answer_text,recall_hit,judge
2,T003,2026년 5월 휴학경험자의 평균 휴학기간은?,1년 9.9개월,모름,True,refused
12,T013,2026년 5월 여자 대졸자의 휴학사유 중 가장 높은 것은?,취업 및 자격시험 준비 (60.8%),취업 및 자격시험준비(55.3%)입니다. [D01 p.29],True,wrong
14,B002,2026년 5월 25~29세 경제활동참가율은?,76.9%,모름,False,refused
15,B003,2026년 5월 20~24세 휴학생 수는?,188천명,모름,False,refused
16,B004,2026년 5월 청년층 졸업 취업자 중 정보통신업 취업자 수는?,183천명,모름,True,refused
23,C001,2022년 5월 15~19세 인구의 전년동월대비 증감은?,-74천명,모름,True,refused
24,C002,2023년 5월 15~19세 인구의 전년동월대비 증감은?,+18천명,모름,False,refused
25,C003,2023년 5월 20~24세 인구의 전년동월대비 증감은?,-110천명,모름,True,refused
45,T024,분기별 콘텐츠산업 동향분석의 표본 사업체 수는?,"2,500개","표본으로 2,500개 사업체를 선정했으며, 표의 조사 표본 수는 2,508개입니다. [D02 p.14] [D02 p.38]",True,wrong


## ⑦ 채점기 검증 준비
채점 모델이 판정한 문항(correct·wrong) 중 20문항을 뽑아 `eval/results/v0_human_check.csv`로 저장합니다.
**LLM 판정은 파일에 넣지 않습니다.** (보고 채점하면 영향을 받으므로)

👉 이 파일의 `human_judge` 칸에 직접 `correct` 또는 `wrong`을 입력합니다.

In [7]:
CHECK_PATH = RESULTS_DIR / "v0_human_check.csv"
# 채점 모델이 판정한 문항만 (모름 문항·'모름' 답변은 규칙으로 판정하므로 제외)
llm_judged = view[(view["type"] != "unknown") & view["judge"].isin(["correct", "wrong"])]
wrong = llm_judged[llm_judged["judge"] == "wrong"]
n_wrong = min(len(wrong), 10)  # 오답을 최대 10개 포함해 판정이 한쪽으로 쏠리지 않게
sample = pd.concat([
    wrong.sample(n=n_wrong, random_state=42),
    llm_judged[llm_judged["judge"] == "correct"].sample(n=20 - n_wrong, random_state=42),
]).sort_values("id")

if CHECK_PATH.exists():
    print("이미 있음 (직접 채점한 내용을 덮어쓰지 않도록 다시 만들지 않음):", CHECK_PATH)
else:
    check = sample[["id", "question", "answer", "answer_text"]].copy()
    check["human_judge"] = ""
    check.to_csv(CHECK_PATH, index=False, encoding="utf-8-sig")  # 엑셀에서 한글이 깨지지 않게
    print("저장:", CHECK_PATH, "| 문항 수:", len(check))

이미 있음 (직접 채점한 내용을 덮어쓰지 않도록 다시 만들지 않음): /home/user/git_project/project6/ReportQA/eval/results/v0_human_check.csv


## ⑧ 일치율 (직접 채점한 뒤 실행)
LLM 판정과 사람 판정이 얼마나 같은지 계산합니다. **목표: 90% 이상**

In [8]:
check = pd.read_csv(CHECK_PATH, dtype=str).fillna("")
filled = check[check["human_judge"].isin(["correct", "wrong"])]
if len(filled) < len(check):
    print(f"아직 채점 전입니다 ({len(filled)}/{len(check)}). human_judge 칸을 채운 뒤 다시 실행하세요.")
else:
    merged = filled.merge(results[["id", "judge", "judge_reason"]], on="id")
    agree = (merged["human_judge"] == merged["judge"]).mean()
    print(f"일치율: {agree:.0%} ({(merged['human_judge'] == merged['judge']).sum()}/{len(merged)})")
    display(merged[merged["human_judge"] != merged["judge"]][["id", "question", "answer", "answer_text", "human_judge", "judge", "judge_reason"]])

일치율: 95% (19/20)


,id,question,answer,answer_text,human_judge,judge,judge_reason
15,T015,2025년 4분기 매출액이 전년 동기 대비 가장 많이 증가한 산업은?,게임 (28.8%),게임 산업입니다. [D02 p.15],wrong,correct,질문에서 요구한 산업이 정답과 동일한 게임 산업입니다.


## ⑨ 발견한 것

### 결과 (2026-11-16, 답변 gpt-6-luna / 채점 gpt-6.1-terra / TOP_K 5 / CHUNK 800·100)

| Recall@5 | 정답률 | 텍스트 | 표 | 차트(vector) | 차트(image) | 환각률 |
|---|---|---|---|---|---|---|
| 87.0% | 67.3% | 92.5% | 53.3% | 61.1% | 0% | 0% |

- 판정: correct 74 / refused(답이 있는데 모름) 28 / wrong 8 / hallucinated 0
- 비용: 답변 276,213 + 15,959 토큰, 채점 23,700 + 3,036 토큰 → 약 $0.075

### 채점기 검증
- LLM이 판정한 문항 중 20개(오답 8 + 정답 12)를 사람이 직접 채점 → **일치율 95% (19/20)** → 목표(90%) 이상이므로 채점기를 그대로 사용
- 불일치 1건 (T015, "매출 증가율 1위 산업"): 답변 "게임 산업"에 정답의 비율(28.8%)이 빠짐 → 사람은 wrong, LLM은 "질문은 산업만 물음"으로 correct
- 판단이 갈릴 만한 T024(정답 2,500개, 답변이 2,508개도 함께 말함)는 사람·LLM 모두 wrong

### 실패 원인 (36문항)
| 원인 | 해당 문항 | 해결할 버전 |
|---|---|---|
| **이미지 차트라 글자가 없음** | 이미지 차트 12개 전부 "모름" (정답 쪽을 찾은 6개 포함) | v2 Vision 캡션 |
| **표를 찾았지만 못 읽음** (열 제목 손실) | 표 실패 14개 중 8개는 정답 쪽 검색 성공 | v1 표 구조 |
| **비슷한 다른 표·연도 값을 가져옴** | 15~29세 질문에 15~34세 통계표(D01 p.28·29·34) 값으로 답함 (T013, B006, B009) / 벡터 차트 숫자가 뒤섞여 다른 연도·선의 값을 고름 (C017, C022, C027, C030) | v1(표 제목 포함) · v2(차트 캡션) |
| **검색 실패** | 정답 쪽이 상위 5개에 없음: 표 7, 이미지 차트 6 | v3 하이브리드 검색·리랭커 |

### 다음 버전에서 볼 것
- v1: 표 정답률(53.3%)과 15~34세 표 혼동이 줄어드는지
- v2: 이미지 차트(0%), 벡터 차트(61.1%)가 오르는지
- 모든 버전에서 환각률 0%가 유지되는지 (정확도를 올리다가 지어내기 시작하면 안 됨)